In [ ]:
#setup
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

np.set_printoptions(precision=4, suppress=True)
pd.options.display.precision = 4
rng = np.random.default_rng(42)
print(sys.executable)

In [ ]:
#four tests on five shapes
datasets = {
    "normal": rng.normal(50, 10, 200),
    "uniform": rng.uniform(0, 100, 200),
    "exponential": rng.exponential(10, 200),
    "t, df=3 (heavy tails)": rng.standard_t(3, 200),
    "lognormal": rng.lognormal(3, 0.6, 200),
}

def normality_report(x):
    x = np.asarray(x)
    ad = stats.anderson(x, dist="norm")
    crit5 = ad.critical_values[list(ad.significance_level).index(5.0)]
    return {"skew": stats.skew(x), "excess kurtosis": stats.kurtosis(x),
            "Shapiro p": stats.shapiro(x).pvalue, "D'Agostino p": stats.normaltest(x).pvalue,
            "Anderson stat": ad.statistic, "Anderson 5% critical": crit5}

report = pd.DataFrame({name: normality_report(x) for name, x in datasets.items()}).T
report["rejects normality?"] = (report["Shapiro p"] < 0.05).map({True: "yes", False: "no"})
print(report.round(4))

In [ ]:
#look at the shape, too
fig, axes = plt.subplots(2, 5, figsize=(16, 5.5))
for col, (name, x) in enumerate(datasets.items()):
    axes[0, col].hist(x, bins=30, color="lightsteelblue", edgecolor="white")
    axes[0, col].set_title(name, fontsize=9)
    stats.probplot(x, dist="norm", plot=axes[1, col])
    axes[1, col].set_title("")
plt.tight_layout()
plt.show()

In [ ]:
#the sample-size paradox
def rejection_rate(draw, n, reps=400):
    return np.mean([stats.shapiro(draw(n)).pvalue < 0.05 for _ in range(reps)])

mild = lambda n: rng.standard_t(10, n)                      # almost normal, with slightly heavy tails
fair = lambda n: rng.normal(0, 1, n)
skewed = lambda n: rng.exponential(1, n)

print(f"{'n':>6} {'normal data':>12} {'almost normal':>14} {'clearly skewed':>15}")
for n in (8, 30, 100, 300, 1000):
    print(f"{n:>6} {rejection_rate(fair, n):>12.2f} {rejection_rate(mild, n):>14.2f} {rejection_rate(skewed, n):>15.2f}")

In [ ]:
#what to do when the data aren’t normal
skewed = rng.lognormal(3, 0.7, 150)
print(f"raw data  : skew {stats.skew(skewed):5.2f}, Shapiro p = {stats.shapiro(skewed).pvalue:.2e}")
print(f"log data  : skew {stats.skew(np.log(skewed)):5.2f}, Shapiro p = {stats.shapiro(np.log(skewed)).pvalue:.3f}")
print(f"sqrt data : skew {stats.skew(np.sqrt(skewed)):5.2f}, Shapiro p = {stats.shapiro(np.sqrt(skewed)).pvalue:.3f}")
bc, lam = stats.boxcox(skewed)
print(f"Box-Cox   : skew {stats.skew(bc):5.2f}, Shapiro p = {stats.shapiro(bc).pvalue:.3f}, lambda = {lam:.2f}")

In [ ]:
#ANOVA by hand
A = np.array([20, 22, 19, 24, 25])
B = np.array([28, 30, 27, 26, 29])
C = np.array([23, 25, 22, 24, 26])
groups = [A, B, C]

def anova_by_hand(groups):
    data = np.concatenate(groups)
    N, k = len(data), len(groups)
    grand = data.mean()
    ssb = sum(len(g) * (g.mean() - grand) ** 2 for g in groups)
    ssw = sum(((g - g.mean()) ** 2).sum() for g in groups)
    df_b, df_w = k - 1, N - k
    msb, msw = ssb / df_b, ssw / df_w
    F = msb / msw
    return {"SSB": ssb, "SSW": ssw, "SST": ssb + ssw, "df between": df_b, "df within": df_w,
            "MSB": msb, "MSW": msw, "F": F, "p": stats.f.sf(F, df_b, df_w), "eta squared": ssb / (ssb + ssw)}

print(pd.Series(anova_by_hand(groups)).round(4))
res = stats.f_oneway(A, B, C)
print("\nscipy:", res)
print("group means:", [g.mean() for g in groups])

In [ ]:
#look at it
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.boxplot(groups)
for i, g in enumerate(groups, start=1):
    ax.plot(rng.normal(i, 0.04, len(g)), g, "o", color="steelblue", alpha=0.7)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["fertiliser A", "fertiliser B", "fertiliser C"])
ax.set_ylabel("yield")
plt.show()

In [ ]:
#why not just run three t-tests?
reps, n = 10_000, 10
g = rng.normal(0, 1, size=(reps, 3, n))                     # three groups with the SAME mean: H0 is true
p_anova = stats.f_oneway(g[:, 0], g[:, 1], g[:, 2], axis=1).pvalue
p_pairs = np.stack([stats.ttest_ind(g[:, i], g[:, j], axis=1).pvalue for i, j in ((0, 1), (0, 2), (1, 2))])

print(f"ANOVA false-positive rate                   : {(p_anova < 0.05).mean():.3f}")
print(f"any of 3 uncorrected t-tests 'significant'  : {(p_pairs.min(axis=0) < 0.05).mean():.3f}")
print(f"any of 3 t-tests with Bonferroni (0.05 / 3) : {(p_pairs.min(axis=0) < 0.05 / 3).mean():.3f}")

In [ ]:
#assumptions, and what to check
resid = np.concatenate([g - g.mean() for g in groups])
print("Shapiro on the residuals:", stats.shapiro(resid))
print("Levene (equal variances):", stats.levene(A, B, C))
print("group variances:", [round(g.var(ddof=1), 2) for g in groups])

In [ ]:
#ANOVA says “some difference”, not “which”
tukey = stats.tukey_hsd(A, B, C)
print(tukey)

ci = tukey.confidence_interval(confidence_level=0.95)
names = ["A", "B", "C"]
for i in range(3):
    for j in range(i + 1, 3):
        print(f"{names[i]} - {names[j]}: difference {A.mean() - B.mean() if (i, j) == (0, 1) else '':>}"
              if False else
              f"{names[i]} - {names[j]}: p = {tukey.pvalue[i, j]:.4f}, 95% CI ({ci.low[i, j]:.2f}, {ci.high[i, j]:.2f})")

msw = anova_by_hand(groups)["MSW"]
q_crit = stats.studentized_range.ppf(0.95, 3, 12)
print(f"\nHSD = q * sqrt(MSW / n) = {q_crit:.3f} * sqrt({msw:.2f} / 5) = {q_crit * math.sqrt(msw / 5):.2f}: "
      "any pair of means further apart than this is significant")

In [ ]:
#the non-parametric option
print(stats.kruskal(A, B, C))
skewed_groups = [rng.lognormal(3, 0.6, 12), rng.lognormal(3.4, 0.6, 12), rng.lognormal(3, 0.6, 12)]
print("ANOVA   :", stats.f_oneway(*skewed_groups).pvalue.round(4))
print("Kruskal :", stats.kruskal(*skewed_groups).pvalue.round(4))

In [ ]:
#ANOVA from a table in long format
df = pd.DataFrame({"fertiliser": ["A"] * 5 + ["B"] * 5 + ["C"] * 5,
                   "yield_": np.concatenate(groups)})
print(df.groupby("fertiliser")["yield_"].agg(["count", "mean", "std"]).round(3))
by_group = [g["yield_"].to_numpy() for _, g in df.groupby("fertiliser")]
print(stats.f_oneway(*by_group))

In [ ]:
#goodness of fit, a die and a pea plant
observed = np.array([25, 17, 15, 23, 24, 16])               # 120 rolls of a die
expected = np.full(6, 20)
chi2 = ((observed - expected) ** 2 / expected).sum()
print(f"by hand: chi2 = {chi2:.3f}, df = 5, p = {stats.chi2.sf(chi2, 5):.4f}")
print("scipy  :", stats.chisquare(observed))                # the default expectation is "all categories equal"

# Mendel's peas: a claimed 9 : 3 : 3 : 1 ratio
peas = np.array([315, 108, 101, 32])
claimed = np.array([9, 3, 3, 1]) / 16 * peas.sum()
print("expected counts:", claimed.round(2))
print("scipy  :", stats.chisquare(peas, f_exp=claimed))

fig, ax = plt.subplots(figsize=(7, 3.4))
width = 0.38
ax.bar(np.arange(4) - width / 2, peas, width, label="observed")
ax.bar(np.arange(4) + width / 2, claimed, width, label="expected under 9:3:3:1")
ax.set_xticks(range(4))
ax.set_xticklabels(["round yellow", "round green", "wrinkled yellow", "wrinkled green"], fontsize=8)
ax.legend()
plt.show()

In [ ]:
#how the statistic behaves when H₀ is true
reps = 20_000
fair = rng.multinomial(120, np.full(6, 1 / 6), size=reps)
stat = ((fair - 20) ** 2 / 20).sum(axis=1)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(stat, bins=50, density=True, color="lightsteelblue", edgecolor="white")
xs = np.linspace(0, 25, 300)
ax.plot(xs, stats.chi2.pdf(xs, 5), color="tomato", lw=2)
ax.set_title("simulated chi-square statistics vs the chi-square(5) curve")
plt.show()
print("share above the 5% critical value:", (stat > stats.chi2.ppf(0.95, 5)).mean().round(3))

In [ ]:
#test of independence, a 2 × 2 table
obs = np.array([[70, 30],                                    # remote: satisfied, not satisfied
                [50, 50]])                                   # office
chi2, p, dof, expected = stats.chi2_contingency(obs, correction=False)
print("expected counts under independence:\n", expected)
print(f"chi2 = {chi2:.3f}, df = {dof}, p = {p:.4f}")

chi2_c, p_c, _, _ = stats.chi2_contingency(obs)              # the default applies Yates' continuity correction to 2x2 tables
print(f"with Yates' correction: chi2 = {chi2_c:.3f}, p = {p_c:.4f}")

odds_ratio, p_fisher = stats.fisher_exact(obs)
print(f"Fisher's exact test: odds ratio = {odds_ratio:.2f}, p = {p_fisher:.4f}")

def cramers_v(table):
    table = np.asarray(table)
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    return math.sqrt(chi2 / (table.sum() * (min(table.shape) - 1)))

print(f"Cramer's V (effect size) = {cramers_v(obs):.3f}")

In [ ]:
#a bigger table, and where the difference lies
table = pd.DataFrame([[30, 45, 25], [40, 35, 25], [20, 30, 50]],
                     index=["Sales", "Support", "Tech"], columns=["Plan A", "Plan B", "Plan C"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.2f}, df = {dof}, p = {p:.5f}, Cramer's V = {cramers_v(table):.3f}")
print("minimum expected count:", expected.min().round(2), "(rule of thumb: at least 5)")

residuals = (table - expected) / np.sqrt(expected)
print("\nstandardised residuals (|value| > 2 stands out):")
print(residuals.round(2))

print("\nrow percentages:")
print((table.div(table.sum(axis=1), axis=0) * 100).round(1))

In [ ]:
#from raw rows to a table, with crosstab
raw = pd.DataFrame({"gender": rng.choice(["F", "M"], 400), "stream": rng.choice(["Arts", "Science", "Commerce"], 400)})
ct = pd.crosstab(raw["gender"], raw["stream"])                # counts for every combination
print(ct)
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"chi2 = {chi2:.2f}, df = {dof}, p = {p:.3f}   (the two columns were generated independently, so a large p is expected)")

In [ ]:
#the sample-size effect, and small tables
base = np.array([[70, 30], [50, 50]])
for factor in (1, 3, 10):
    t = base * factor
    chi2, p, _, _ = stats.chi2_contingency(t, correction=False)
    print(f"counts x {factor:>2}: chi2 = {chi2:6.2f}, p = {p:.2e}, Cramer's V = {cramers_v(t):.3f}")

print("\nsmall table (expected counts are 2, below 5):")
tiny = np.array([[3, 1], [1, 3]])
print(" chi-square p :", round(stats.chi2_contingency(tiny)[1], 4), "(unreliable)")
print(" Fisher exact :", round(stats.fisher_exact(tiny)[1], 4), "(valid for any size)")

In [ ]:
#by hand
x = np.array([1, 2, 3, 4, 5])
y = np.array([2, 4, 5, 4, 5])
dx, dy = x - x.mean(), y - y.mean()
print("products of deviations:", dx * dy)
cov = (dx * dy).sum() / (len(x) - 1)
r = cov / (x.std(ddof=1) * y.std(ddof=1))
print(f"covariance = {cov:.4f}, r = {r:.4f}")

print("\nnp.cov returns a MATRIX (variances on the diagonal):\n", np.cov(x, y))
print("np.corrcoef:\n", np.corrcoef(x, y))
print("population covariance (ddof=0):", np.cov(x, y, ddof=0)[0, 1])
print("pearsonr:", stats.pearsonr(x, y))

In [ ]:
#covariance depends on units, and correlation doesn’t
height_cm = rng.normal(170, 10, 500)
weight_kg = 0.9 * height_cm - 85 + rng.normal(0, 8, 500)
print(f"height in cm : cov = {np.cov(height_cm, weight_kg)[0, 1]:8.2f}, r = {np.corrcoef(height_cm, weight_kg)[0, 1]:.4f}")
print(f"height in m  : cov = {np.cov(height_cm / 100, weight_kg)[0, 1]:8.4f}, r = {np.corrcoef(height_cm / 100, weight_kg)[0, 1]:.4f}")
print(f"height in inches, weight in lb: r = {np.corrcoef(height_cm / 2.54, weight_kg * 2.2046)[0, 1]:.4f}")

In [ ]:
#what r looks like
def make_pair(r, n=300):
    x = rng.normal(size=n)
    return x, r * x + math.sqrt(1 - r ** 2) * rng.normal(size=n)

fig, axes = plt.subplots(2, 4, figsize=(15, 6.4))
for ax, target in zip(axes[0], (0.95, 0.7, 0.3, 0.0)):
    x, y = make_pair(target)
    ax.scatter(x, y, s=8, alpha=0.6)
    ax.set_title(f"r = {np.corrcoef(x, y)[0, 1]:.2f}")
x, y = make_pair(-0.8)
axes[1, 0].scatter(x, y, s=8, alpha=0.6)
axes[1, 0].set_title(f"negative: r = {np.corrcoef(x, y)[0, 1]:.2f}")

x = rng.uniform(-3, 3, 300)
y = x ** 2 + rng.normal(0, 0.5, 300)
axes[1, 1].scatter(x, y, s=8, alpha=0.6)
axes[1, 1].set_title(f"perfect U-shape: r = {np.corrcoef(x, y)[0, 1]:.2f}")

x, y = make_pair(0.0, 40)
x2, y2 = np.append(x, 8), np.append(y, 8)
axes[1, 2].scatter(x2, y2, s=10)
axes[1, 2].set_title(f"ONE outlier: r = {np.corrcoef(x2, y2)[0, 1]:.2f} (was {np.corrcoef(x, y)[0, 1]:.2f})")

x, y = make_pair(0.8, 3000)
keep = x > 1
axes[1, 3].scatter(x[keep], y[keep], s=8, alpha=0.6)
axes[1, 3].set_title(f"restricted range: r = {np.corrcoef(x[keep], y[keep])[0, 1]:.2f} (from 0.80)")
plt.tight_layout()
plt.show()

In [ ]:
#Anscombe’s quartet, a lesson that never gets old
x123 = np.array([10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5])
quartet = {
    "I": (x123, np.array([8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68])),
    "II": (x123, np.array([9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74])),
    "III": (x123, np.array([7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73])),
    "IV": (np.array([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8]),
           np.array([6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89])),
}
rows = {}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, (name, (x, y)) in zip(axes, quartet.items()):
    fit = stats.linregress(x, y)
    rows[name] = {"mean x": x.mean(), "mean y": y.mean(), "r": fit.rvalue, "slope": fit.slope, "intercept": fit.intercept}
    ax.scatter(x, y)
    xs = np.array([3, 20])
    ax.plot(xs, fit.intercept + fit.slope * xs, color="tomato")
    ax.set_title(f"dataset {name}")
plt.tight_layout()
plt.show()
print(pd.DataFrame(rows).T.round(3))

In [ ]:
#testing and measuring r
x, y = make_pair(0.4, 50)
r, p = stats.pearsonr(x, y)
print(f"n = 50  : r = {r:.3f}, p = {p:.4f}, r squared = {r ** 2:.3f}")

t = r * math.sqrt(50 - 2) / math.sqrt(1 - r ** 2)
print(f"by hand : t = {t:.3f}, df = 48, p = {2 * stats.t.sf(abs(t), 48):.4f}")

def pearson_with_ci(x, y, level=0.95):
    r, p = stats.pearsonr(x, y)
    z, se = np.arctanh(r), 1 / math.sqrt(len(x) - 3)           # Fisher's z transform
    crit = stats.norm.ppf((1 + level) / 2)
    return r, p, (np.tanh(z - crit * se), np.tanh(z + crit * se))

print("95% CI for r:", np.round(pearson_with_ci(x, y)[2], 3))

x, y = make_pair(0.08, 3000)
r, p, ci = pearson_with_ci(x, y)
print(f"n = 3000: r = {r:.3f}, p = {p:.4f}, r squared = {r ** 2:.4f}  -> 'significant', but explains under 1% of the variation")

In [ ]:
#Pearson, Spearman, Kendall
x = np.arange(1, 11)
y = x ** 3                                                     # perfectly monotone, but curved
print(f"cubic curve: Pearson {stats.pearsonr(x, y)[0]:.3f}, Spearman {stats.spearmanr(x, y).statistic:.3f}, "
      f"Kendall {stats.kendalltau(x, y).statistic:.3f}")

y_out = y.copy()
y_out[-1] = 5000
print(f"one wild point: Pearson {stats.pearsonr(x, y_out)[0]:.3f}, Spearman {stats.spearmanr(x, y_out).statistic:.3f}")

ranks_x, ranks_y = stats.rankdata(x), stats.rankdata(y_out)
print("Spearman is Pearson on the ranks:", round(stats.pearsonr(ranks_x, ranks_y)[0], 3))

In [ ]:
#a correlation matrix
n = 400
hours = rng.gamma(4, 1.2, n)
sleep = np.clip(rng.normal(7, 1, n), 4, 10)
screen = np.clip(6 - 0.3 * hours + rng.normal(0, 1.5, n), 0, 12)
marks = np.clip(40 + 3 * hours + 2 * (sleep - 7) - 0.8 * screen + rng.normal(0, 8, n), 0, 100)
data = pd.DataFrame({"hours": hours, "sleep": sleep, "screen": screen, "marks": marks})

corr = data.corr()                                             # Pearson by default; use method="spearman" for ranks
print(corr.round(2))
print("\ncovariance matrix:\n", data.cov().round(2))

fig, ax = plt.subplots(figsize=(5.5, 4.6))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im)
plt.tight_layout()
plt.show()

In [ ]:
#a confounder
temp = rng.normal(25, 6, 365)                                  # daily temperature
ice_cream = 20 + 3 * temp + rng.normal(0, 6, 365)              # sales rise with heat
drownings = 1 + 0.2 * temp + rng.normal(0, 1.2, 365)           # swimming rises with heat; ice cream plays no part

def residuals(y, x):
    slope, intercept = np.polyfit(x, y, 1)
    return y - (slope * x + intercept)

print(f"ice cream vs drownings          : r = {np.corrcoef(ice_cream, drownings)[0, 1]:.2f}")
print(f"after removing the temperature  : r = {np.corrcoef(residuals(ice_cream, temp), residuals(drownings, temp))[0, 1]:.2f}   (a partial correlation)")

In [ ]:
#Simpson’s paradox
frames = []
for g in range(3):
    xg = 10 * g + rng.normal(0, 2, 100)
    yg = 15 * g - 1.0 * (xg - 10 * g) + rng.normal(0, 1, 100)        # inside each group, Y FALLS as X rises
    frames.append(pd.DataFrame({"group": g, "x": xg, "y": yg}))
sim = pd.concat(frames)

print("overall correlation:", round(sim["x"].corr(sim["y"]), 2))
print("within each group  :", sim.groupby("group").apply(lambda d: d["x"].corr(d["y"]), include_groups=False).round(2).tolist())

fig, ax = plt.subplots(figsize=(6, 3.8))
for g, d in sim.groupby("group"):
    ax.scatter(d["x"], d["y"], s=10, label=f"group {g}")
ax.legend()
plt.show()

#the four pre-registered questions
1. Do mean marks differ between the three teaching methods?        (ANOVA + Tukey, or Kruskal)
2. Is the pass rate related to the department?                     (chi-square independence)
3. How strongly are study hours and marks related?                 (Pearson / Spearman, with a CI)
4. Is the method effect still there once study hours are considered? (partial correlation idea, S5 Cell 26)

In [ ]:
#the analysis
# Q1: marks by method
by_method = [d["marks"].to_numpy() for _, d in students.groupby("method")]
print("1. group means:", students.groupby("method")["marks"].mean().round(1).to_dict())
resid = np.concatenate([g - g.mean() for g in by_method])
print("   residual normality (Shapiro):", round(stats.shapiro(resid).pvalue, 3), "| Levene:", round(stats.levene(*by_method).pvalue, 3))
anova = stats.f_oneway(*by_method)
total_ss = ((students["marks"] - students["marks"].mean()) ** 2).sum()
between_ss = sum(len(g) * (g.mean() - students["marks"].mean()) ** 2 for g in by_method)
print(f"   ANOVA: F = {anova.statistic:.2f}, p = {anova.pvalue:.4f}, eta squared = {between_ss / total_ss:.3f}")
print(stats.tukey_hsd(*by_method))

# Q2: pass rate by department
ct = pd.crosstab(students["dept"], students["passed"])
chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f"\n2. chi2 = {chi2:.2f}, df = {dof}, p = {p:.3f}, Cramer's V = {cramers_v(ct):.3f}, min expected = {expected.min():.1f}")
print(ct)

# Q3: hours and marks
r, p, ci = pearson_with_ci(students["hours"], students["marks"])
print(f"\n3. Pearson r = {r:.3f} (95% CI {ci[0]:.3f} to {ci[1]:.3f}), r squared = {r ** 2:.3f}, "
      f"Spearman = {stats.spearmanr(students['hours'], students['marks']).statistic:.3f}")

# Q4: remove the effect of study hours, then compare the methods again
students["marks_adj"] = residuals(students["marks"].to_numpy(), students["hours"].to_numpy())
adj_groups = [d["marks_adj"].to_numpy() for _, d in students.groupby("method")]
print(f"\n4. after adjusting for hours: ANOVA p = {stats.f_oneway(*adj_groups).pvalue:.4f}")

the conclusion
## Findings

**Teaching methods.** Mean marks were __ (A), __ (B), __ (C). ANOVA: F = __, p = __, which means __.
Tukey: only __ differs from __ (difference __ marks, 95% CI __ to __).

**Department and passing.** The pass rate was __ in Arts, __ in Science and __ in Commerce.
Chi-square p = __ (Cramer's V = __), so __.

**Study hours.** r = __ (95% CI __ to __): each extra hour is associated with about __ more marks.
This is an association; it does not prove that studying causes the marks. (Name a possible confounder.)

**Limits.** (Four tests were planned in advance; observational data; the simulation assumed ... .)
